In [ ]:
from sklearn.decomposition import PCA
def pca(df_X, df_y):
    pca = PCA(n_components = 2)
    pca.fit(df_X)
    df_pca = pca.transform(df_X)
    df_pca = pd.DataFrame(df_pca, columns = ['comp. 0', 'comp. 1'])
    df_pca['target'] = df_y
    print('variance ratio:', pca.explained_variance_ratio_, 'sum:', sum(pca.explained_variance_ratio_))
    return df_pca

def concath(df_X, df_y):
    df = pd.concat([df_X, df_y])
    return df

In [ ]:
from keras.models import Sequential
from keras.layers import Dense
from tensorflow.keras.optimizers import SGD
from matplotlib import pyplot as plt
from sklearn import metrics
import tensorflow as tf
import keras
from keras.layers import BatchNormalization
from keras.layers import Activation
from keras import optimizers
import math


################################ MSE ################################
def MSE(y_true, y_pred):
    return tf.reduce_mean(tf.math.square(y_true - y_pred))

################################ BCE ################################
import tensorflow as tf
def BCE(y_true, y_pred):
    return -tf.reduce_mean(y_true*tf.math.log(y_pred)+(1-y_true)*tf.math.log(1-y_pred))

################################ Ours_Accu ################################
def Ours_Accu(y_true, y_pred):
    y_pred = 1/(1+tf.math.exp(-L*(y_pred-0.5)))
    yl = y_train.shape[0]
    accu = (yl-tf.reduce_sum(y_true)-tf.reduce_sum(y_pred)+2*tf.reduce_sum(y_true*y_pred)) / yl
    return 1-accu

################################ Ours_Fbeta ################################
def Ours_Fbeta(y_true, y_pred):
#     beta = 1 
    y_pred = 1/(1+tf.math.exp(-L*(y_pred-0.5)))
    numerator = (1+beta**2)*tf.reduce_sum(y_true*y_pred)
    denominator = (beta**2)*tf.reduce_sum(y_true) + tf.reduce_sum(y_pred)
    return 1-(numerator/denominator)

################################ Ours_Gmean ################################
def Ours_Gmean(y_true, y_pred):
    y_pred = 1/(1+tf.math.exp(-L*(y_pred-0.5)))
    syhy = tf.reduce_sum(y_true*y_pred)
    sy = tf.reduce_sum(y_true)
    yl = (y_train.shape[0])
#     gmean = syhy*(yl-tf.reduce_sum(y_pred)-sy+syhy)/(sy*(yl-sy))
    gmean = tf.sqrt(syhy*(yl-tf.reduce_sum(y_pred)-sy+syhy)/(sy*(yl-sy)))
    return 1-gmean

################################ Ours_BAccu ################################
def Ours_BAccu(y_true, y_pred):
    y_pred = 1/(1+tf.math.exp(-L*(y_pred-0.5)))
    syhy = tf.reduce_sum(y_true*y_pred)
    sy = tf.reduce_sum(y_true)
    yl = y_train.shape[0]
    baccu = (yl*(syhy+sy)-sy*(tf.reduce_sum(y_pred)+sy)) / (2*sy*(yl-sy))
    return 1-baccu


# 1. My own data(2d / 10,000)

In [ ]:
from sklearn import datasets
import numpy as np
import pandas as pd
Init_X, Init_y = datasets.make_classification(n_samples=10000, n_classes=2, weights=[0.9, 0.1], class_sep=1.2,
                                    n_features=5, n_informative=3, n_redundant=1, n_clusters_per_class=1, random_state=0)
X = np.array(Init_X)
y = np.array(Init_y)
# # change 0 -> -1
# y = [-1 if x==0 else x for x in y]

df_pca = pca(X, y)
df_pca

In [ ]:
L = 73
hidden_node = 2
# momentum=0.9
activation = 'sigmoid'  
kernel_initializer=keras.initializers.he_normal(seed=100)
epochs=50
threshold = 0.5

In [ ]:
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state = 2)

fscore = []

X = df_pca.iloc[:, :2]
y = df_pca.iloc[:, 2]

for i in range(34):
    print('!'*50,'L={0}'.format(3*i+1),'!'*50)
    L = 3*i + 1
    f1 = []
    batch_size = int(X.shape[0]*0.9 * 0.05)  
    learning_rate = 0.001
    model = Sequential()
    model.add(Dense(hidden_node, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
    model.add(BatchNormalization())
    model.add(Activation(activation))
    model.add(Dense(1))
    opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)
    
    n_iter=0
    for train_index, test_index in skf.split(X, y):
        n_iter += 1
        X_train = X.iloc[train_index]
        y_train= y.iloc[train_index]
        X_test = X.iloc[test_index]
        y_test= y.iloc[test_index]
    #     print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
        X_train = np.array(X_train)
        y_train = np.array(y_train)
        y_train = y_train.astype(float)
        X_test = np.array(X_test)
        y_test = np.array(y_test)
        y_test = y_test.astype(float)

        ###################### Ours(F1) ##############################  
        beta = 1
        model.compile(loss=Ours_Fbeta, optimizer=opt, metrics=['accuracy'])
        history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
#         plt.plot(history.history['loss'], label='loss')
#         plt.ylim([0, 1])
#         plt.xlabel('Iteration',fontweight="bold",fontsize = 15)
#         plt.ylabel('Loss',fontweight="bold",fontsize = 15)
#         plt.title("Cost Function",fontweight="bold",fontsize = 20)
#         plt.legend()
#         plt.show()
        predicted = []
        result = model.predict(X_test)
        for i in range(X_test.shape[0]):
            if result[i] <= threshold:
                predicted.append(0)
            else:
                predicted.append(1)
        TN = metrics.confusion_matrix(y_test, predicted)[0,0]
        FP = metrics.confusion_matrix(y_test, predicted)[0,1]
        FN = metrics.confusion_matrix(y_test, predicted)[1,0]
        TP = metrics.confusion_matrix(y_test, predicted)[1,1]
        f1.append(TP / (TP + 0.5*(FP+FN)))

    print('F1 =', np.mean(f1), f1)
    fscore.append(np.mean(f1))

In [ ]:
df_f = pd.DataFrame({'L':list(np.arange(1, 101, 3)), 'fscore':fscore})
df_f

In [ ]:
x = df_f['L']
y = df_f['fscore']

plt.figure(figsize=(12,10))
plt.plot(x,y,linewidth=3, label = 'F1-score')

# plt.ylim(0.0,0.6) # y축 값의 범위 설정
plt.legend()
plt.show()

In [ ]:
df_f.to_csv("5CV_MLP_Random_L.csv", float_format='%.4g')

In [ ]:
df = pd.read_csv("5CV_MLP_Random_L.csv")
df

In [ ]:
# plt.rcParams["figure.figsize"] = [5, 3]
plt.rcParams["figure.dpi"] = 200
plt.rcParams['axes.titlesize'] = 20  
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['axes.labelsize'] = 20  
plt.rcParams['font.size'] = 15

x = df['L']
y = df['fscore']

plt.plot(x,y,linewidth=3, label = 'F1-score')

plt.ylim(0.1,0.7) # y축 값의 범위 설정
plt.xlabel('L',fontsize = 20)
plt.ylabel('Score',fontsize = 20)
plt.title("F1_Dataset#1",fontsize = 20)
# plt.legend()
plt.show()

# 2. Creditcard Fraud Detection 2023(29d / 298531)

In [ ]:
# class '0' = normal, class '1' = anomaly
card_df = pd.read_csv('creditcard_2023.csv')
card_df.shape

In [ ]:
card_df.isnull().sum()

In [ ]:
card_df.head()

In [ ]:
card_df.describe()

In [ ]:
# Amount values largely varies.

# # Normalization
# card_df.iloc[:,:-1] = (card_df.iloc[:,:-1] - card_df.iloc[:,:-1].min())/(card_df.iloc[:,:-1].max() - card_df.iloc[:,:-1].min())

# Standardization
card_df.iloc[:,:-1] = (card_df.iloc[:,:-1] - card_df.iloc[:,:-1].mean())/card_df.iloc[:,:-1].std()

card_df

In [ ]:
card_df['Class'].value_counts()

In [ ]:
# Data is too balanced!!! We intentionally make it imbalanced.
df_0 = card_df[card_df['Class']==0]
df_1 = card_df[card_df['Class']==1]
print(len(df_0), len(df_1))

In [ ]:
N = round(len(df_0)*0.05)
df_1_samp = df_1.sample(n=N, random_state = 100)
df_1_samp

In [ ]:
df_card = concath(df_0, df_1_samp)
df_card

In [ ]:
df_card.columns

In [ ]:
df_card = df_card.drop('id', axis=1)
df_card

In [ ]:
L = 73
hidden_node = 2
# momentum=0.9
activation = 'sigmoid'  
kernel_initializer=keras.initializers.he_normal(seed=100)
epochs=50
threshold = 0.5

In [ ]:
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state = 2)

fscore = []

X = df_card.iloc[:, :-1]
y = df_card.iloc[:, -1]

for i in range(34):
    print('!'*50,'L={0}'.format(3*i+1),'!'*50)
    L = 3*i + 1
    f1 = []
    batch_size = int(X.shape[0]*0.9 * 0.05)  
    learning_rate = 0.005
    model = Sequential()
    model.add(Dense(hidden_node, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
    model.add(BatchNormalization())
    model.add(Activation(activation))
    model.add(Dense(1))
    opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)
    
    n_iter=0
    for train_index, test_index in skf.split(X, y):
        n_iter += 1
        X_train = X.iloc[train_index]
        y_train= y.iloc[train_index]
        X_test = X.iloc[test_index]
        y_test= y.iloc[test_index]
    #     print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
        X_train = np.array(X_train)
        y_train = np.array(y_train)
        y_train = y_train.astype(float)
        X_test = np.array(X_test)
        y_test = np.array(y_test)
        y_test = y_test.astype(float)

        ###################### Ours(F1) ##############################  
        beta = 1
        model.compile(loss=Ours_Fbeta, optimizer=opt, metrics=['accuracy'])
        history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
#         plt.plot(history.history['loss'], label='loss')
#         plt.ylim([0, 1])
#         plt.xlabel('Iteration',fontweight="bold",fontsize = 15)
#         plt.ylabel('Loss',fontweight="bold",fontsize = 15)
#         plt.title("Cost Function",fontweight="bold",fontsize = 20)
#         plt.legend()
#         plt.show()
        predicted = []
        result = model.predict(X_test)
        for i in range(X_test.shape[0]):
            if result[i] <= threshold:
                predicted.append(0)
            else:
                predicted.append(1)
        TN = metrics.confusion_matrix(y_test, predicted)[0,0]
        FP = metrics.confusion_matrix(y_test, predicted)[0,1]
        FN = metrics.confusion_matrix(y_test, predicted)[1,0]
        TP = metrics.confusion_matrix(y_test, predicted)[1,1]
        f1.append(TP / (TP + 0.5*(FP+FN)))

    print('F1 =', np.mean(f1), f1)
    fscore.append(np.mean(f1))

In [ ]:
df_f = pd.DataFrame({'L':list(np.arange(1, 101, 3)), 'fscore':fscore})
df_f

In [ ]:
x = df['L']
y = df_f['fscore']

plt.figure(figsize=(12,10))
plt.plot(x,y,linewidth=3, label = 'F1-score')

# plt.ylim(0.6,0.65) # y축 값의 범위 설정
plt.legend()
plt.show()

In [ ]:
df_f.to_csv("5CV_MLP_Card_L.csv", float_format='%.4g')

In [ ]:
df = pd.read_csv("5CV_MLP_Card_L.csv")
df

In [ ]:
# plt.rcParams["figure.figsize"] = [5, 3]
plt.rcParams["figure.dpi"] = 200
plt.rcParams['axes.titlesize'] = 20  
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['axes.labelsize'] = 20  
plt.rcParams['font.size'] = 15

x = df['L']
y = df['fscore']

plt.plot(x,y,linewidth=3, label = 'F1-score')

# plt.ylim(0.1,0.9) # y축 값의 범위 설정
plt.xlabel('L',fontsize = 20)
plt.ylabel('Score',fontsize = 20)
plt.title("F1_Dataset#2",fontsize = 20)
# plt.legend()
plt.show()

# 3. Breast Cancer Data (30d / 569)

In [ ]:
# class 'B' = Benign, class 'M' = Malignant
cancer_df = pd.read_csv('breast_cancer.csv')
cancer_df.shape

In [ ]:
cancer_df.isnull().sum()

In [ ]:
cancer_df.head()

In [ ]:
cancer_df.describe()

In [ ]:
# M/Malignant = 0, B/Benign = 1
y_encoded, y_class = pd.factorize(cancer_df['diagnosis'])
print(y_class)
y_encoded

In [ ]:
# But I want [B/Benign = 0(Major), M/Malignant = 1(minor)]
y_encoded = (y_encoded+1)%2
y_encoded

In [ ]:
cancer_df['label'] = y_encoded
cancer_df

In [ ]:
cancer_df = cancer_df.drop('id', axis=1)
cancer_df = cancer_df.drop('diagnosis', axis=1)
cancer_df

In [ ]:
# Amount values largely varies.

# # Normalization
# card_df.iloc[:,:-1] = (card_df.iloc[:,:-1] - card_df.iloc[:,:-1].min())/(card_df.iloc[:,:-1].max() - card_df.iloc[:,:-1].min())

# Standardization
cancer_df.iloc[:,:-1] = (cancer_df.iloc[:,:-1] - cancer_df.iloc[:,:-1].mean())/cancer_df.iloc[:,:-1].std()

cancer_df

In [ ]:
cancer_df['label'].value_counts()

In [ ]:
# Data is too balanced!!! We intentionally make it imbalanced.
df_0 = cancer_df[cancer_df['label']==0]
df_1 = cancer_df[cancer_df['label']==1]
print(len(df_0), len(df_1))

In [ ]:
N = round(len(df_0)*0.1)
df_1_samp = df_1.sample(n=N, random_state = 100)
df_1_samp

In [ ]:
cancer_df = concath(df_0, df_1_samp)
cancer_df

In [ ]:
cancer_df['label'].value_counts()

In [ ]:
L = 73
hidden_node = 2
# momentum=0.9
activation = 'sigmoid'  
kernel_initializer=keras.initializers.he_normal(seed=100)
epochs=50
threshold = 0.5

In [ ]:
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state = 2)

fscore = []

X = cancer_df.iloc[:, :-1]
y = cancer_df.iloc[:, -1]

for i in range(34):
    print('!'*50,'L={0}'.format(3*i+1),'!'*50)
    L = 3*i + 1
    f1 = []
    batch_size = int(X.shape[0]*0.9 * 0.05)  
    learning_rate = 0.01
    model = Sequential()
    model.add(Dense(hidden_node, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
    model.add(BatchNormalization())
    model.add(Activation(activation))
    model.add(Dense(1))
    opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)
    
    n_iter=0
    for train_index, test_index in skf.split(X, y):
        n_iter += 1
        X_train = X.iloc[train_index]
        y_train= y.iloc[train_index]
        X_test = X.iloc[test_index]
        y_test= y.iloc[test_index]
    #     print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
        X_train = np.array(X_train)
        y_train = np.array(y_train)
        y_train = y_train.astype(float)
        X_test = np.array(X_test)
        y_test = np.array(y_test)
        y_test = y_test.astype(float)

        ###################### Ours(F1) ##############################  
        beta = 1
        model.compile(loss=Ours_Fbeta, optimizer=opt, metrics=['accuracy'])
        history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
#         plt.plot(history.history['loss'], label='loss')
#         plt.ylim([0, 1])
#         plt.xlabel('Iteration',fontweight="bold",fontsize = 15)
#         plt.ylabel('Loss',fontweight="bold",fontsize = 15)
#         plt.title("Cost Function",fontweight="bold",fontsize = 20)
#         plt.legend()
#         plt.show()
        predicted = []
        result = model.predict(X_test)
        for i in range(X_test.shape[0]):
            if result[i] <= threshold:
                predicted.append(0)
            else:
                predicted.append(1)
        TN = metrics.confusion_matrix(y_test, predicted)[0,0]
        FP = metrics.confusion_matrix(y_test, predicted)[0,1]
        FN = metrics.confusion_matrix(y_test, predicted)[1,0]
        TP = metrics.confusion_matrix(y_test, predicted)[1,1]
        f1.append(TP / (TP + 0.5*(FP+FN)))

    print('F1 =', np.mean(f1), f1)
    fscore.append(np.mean(f1))

In [ ]:
df_f = pd.DataFrame({'L':list(np.arange(1, 101, 3)), 'fscore':fscore})
df_f

In [ ]:
x = df['L']
y = df_f['fscore']

plt.figure(figsize=(12,10))
plt.plot(x,y,linewidth=3, label = 'F1-score')

# plt.ylim(0.6,0.65) # y축 값의 범위 설정
plt.legend()
plt.show()

In [ ]:
df_f.to_csv("5CV_MLP_Cancer_L.csv", float_format='%.4g')

In [ ]:
df = pd.read_csv("5CV_MLP_Cancer_L.csv")
df

In [ ]:
# plt.rcParams["figure.figsize"] = [5, 3]
plt.rcParams["figure.dpi"] = 200
plt.rcParams['axes.titlesize'] = 20  
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['axes.labelsize'] = 20  
plt.rcParams['font.size'] = 15

x = df['L']
y = df['fscore']

plt.plot(x,y,linewidth=3, label = 'F1-score')

plt.ylim(0.1,1.0) # y축 값의 범위 설정
plt.xlabel('L',fontsize = 20)
plt.ylabel('Score',fontsize = 20)
plt.title("F1_Dataset#3",fontsize = 20)
# plt.legend()
plt.show()

# 4. Diabetes Prediction Data (8d / 100000)

In [ ]:
# class 'B' = Benign, class 'M' = Malignant
diab_df = pd.read_csv('diabetes_prediction_dataset.csv')
diab_df.shape

In [ ]:
diab_df.isnull().sum()

In [ ]:
diab_df

In [ ]:
# Female = 0, Male = 1, other = 2
gen_encoded, gen_class = pd.factorize(diab_df['gender'])
print(gen_class)
gen_encoded

In [ ]:
# Female = 0, Male = 1, other = 2
pd.Series(gen_encoded).value_counts()

In [ ]:
diab_df['gender'] = gen_encoded
diab_df

In [ ]:
# never = 0, Info = 1, current = 2, former=3, ever=4, not current=5
smo_encoded, smo_class = pd.factorize(diab_df['smoking_history'])
print(smo_class)
smo_encoded

In [ ]:
# never = 0, Info = 1, current = 2, former=3, ever=4, not current=5
pd.Series(smo_encoded).value_counts()

In [ ]:
diab_df['smoking_history'] = smo_encoded
diab_df

In [ ]:
diab_df.describe()

In [ ]:
# Amount values largely varies.

# # Normalization
# card_df.iloc[:,:-1] = (card_df.iloc[:,:-1] - card_df.iloc[:,:-1].min())/(card_df.iloc[:,:-1].max() - card_df.iloc[:,:-1].min())

# Standardization
diab_df.iloc[:,:-1] = (diab_df.iloc[:,:-1] - diab_df.iloc[:,:-1].mean())/diab_df.iloc[:,:-1].std()

diab_df

In [ ]:
L = 73
hidden_node = 2
# momentum=0.9
activation = 'sigmoid'  
kernel_initializer=keras.initializers.he_normal(seed=100)
epochs=50
threshold = 0.5

In [ ]:
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state = 2)

fscore = []

X = diab_df.iloc[:, :-1]
y = diab_df.iloc[:, -1]

for i in range(34):
    print('!'*50,'L={0}'.format(3*i+1),'!'*50)
    L = 3*i + 1
    f1 = []
    batch_size = int(X.shape[0]*0.9 * 0.05)  
    learning_rate = 0.003
    model = Sequential()
    model.add(Dense(hidden_node, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
    model.add(BatchNormalization())
    model.add(Activation(activation))
    model.add(Dense(1))
    opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)
    
    n_iter=0
    for train_index, test_index in skf.split(X, y):
        n_iter += 1
        X_train = X.iloc[train_index]
        y_train= y.iloc[train_index]
        X_test = X.iloc[test_index]
        y_test= y.iloc[test_index]
    #     print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
        X_train = np.array(X_train)
        y_train = np.array(y_train)
        y_train = y_train.astype(float)
        X_test = np.array(X_test)
        y_test = np.array(y_test)
        y_test = y_test.astype(float)

        ###################### Ours(F1) ##############################  
        beta = 1
        model.compile(loss=Ours_Fbeta, optimizer=opt, metrics=['accuracy'])
        history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
#         plt.plot(history.history['loss'], label='loss')
#         plt.ylim([0, 1])
#         plt.xlabel('Iteration',fontweight="bold",fontsize = 15)
#         plt.ylabel('Loss',fontweight="bold",fontsize = 15)
#         plt.title("Cost Function",fontweight="bold",fontsize = 20)
#         plt.legend()
#         plt.show()
        predicted = []
        result = model.predict(X_test)
        for i in range(X_test.shape[0]):
            if result[i] <= threshold:
                predicted.append(0)
            else:
                predicted.append(1)
        TN = metrics.confusion_matrix(y_test, predicted)[0,0]
        FP = metrics.confusion_matrix(y_test, predicted)[0,1]
        FN = metrics.confusion_matrix(y_test, predicted)[1,0]
        TP = metrics.confusion_matrix(y_test, predicted)[1,1]
        f1.append(TP / (TP + 0.5*(FP+FN)))

    print('F1 =', np.mean(f1), f1)
    fscore.append(np.mean(f1))

In [ ]:
df_f = pd.DataFrame({'L':list(np.arange(1, 101, 3)), 'fscore':fscore})
df_f

In [ ]:
x = df['L']
y = df_f['fscore']

plt.figure(figsize=(12,10))
plt.plot(x,y,linewidth=3, label = 'F1-score')

# plt.ylim(0.6,0.65) # y축 값의 범위 설정
plt.legend()
plt.show()

In [ ]:
df_f.to_csv("5CV_MLP_Diab_L.csv", float_format='%.4g')

In [ ]:
df = pd.read_csv("5CV_MLP_Diab_L.csv")
df

In [ ]:
# plt.rcParams["figure.figsize"] = [5, 3]
plt.rcParams["figure.dpi"] = 200
plt.rcParams['axes.titlesize'] = 20  
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['axes.labelsize'] = 20  
plt.rcParams['font.size'] = 15

x = df['L']
y = df['fscore']

plt.plot(x,y,linewidth=3, label = 'F1-score')

plt.ylim(0.1,0.8) # y축 값의 범위 설정
plt.xlabel('L',fontsize = 20)
plt.ylabel('Score',fontsize = 20)
plt.title("F1_Dataset#4",fontsize = 20)
# plt.legend()
plt.show()